# ML-03 — Frame Your Lane as an ML Task

## 1. My lane as an ML task (type)

**Lane: Refresh / Content Opportunity Scoring.**

I frame this as a **ranking / scoring** problem. The practical question is not simply whether a page is declining; it is **which pages should be reviewed first** when an SEO/content team has limited review capacity. The output would be a priority score that orders pages from higher to lower review opportunity. The FlyRank framing guide maps "Which ones first?" to ranking/scoring and identifies precision@K as a typical metric.

This remains a provisional framing. Later weeks may refine the target and time window once the signal audit and data contract are clearer.


In [ ]:
# State the task framing before modeling.
task_type = 'ranking / scoring'
decision = 'Which content pages should be reviewed first?'
print(f'Task type: {task_type}')
print(f'Decision: {decision}')


## 2. Target or proxy

For the starter playground, the available proxy is **`is_declining_label = (trend_direction == 'down')`**. This is a defined current-window proxy, not a true future outcome. I will use it only as a starting point for understanding the task and will not describe it as proof of future decline.

For the stronger capstone version, I would prefer an observed **future-window outcome**, for example whether a page's traffic or search performance declines during a clearly defined period after the decision point. That would better match the real decision because the features would represent information available before the outcome.

The target must be kept separate from the features: `trend_direction` and `trend_pct` must not be model features when they define the starter proxy.


In [ ]:
# Sketch the starter proxy target.
# It is intentionally derived from the observed current-window field.
df['is_declining_proxy'] = (df['trend_direction'] == 'down').astype(int)
print(df['is_declining_proxy'].value_counts(dropna=False).sort_index())


## 3. Success metric

**Primary metric: Precision@K.**

K represents the number of pages the team can realistically review in a batch. Precision@K answers the decision question directly: among the top K pages in the ranked queue, how many are actually positive under the chosen target?

I prefer this to a generic accuracy score because the operational decision is about the quality of the **top of the queue**, not about classifying every page equally. The exact K should be chosen to match a realistic review capacity and declared before evaluation.


In [ ]:
K = 50
print(f'Primary evaluation metric: Precision@{K}')
print('Interpretation: quality of the top-K review queue.')


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one content page (one pseudonymized content item) per row.**

For the starter dataset, the file contains 30,000 rows and 44 columns. I will inspect the page-level rows directly and keep identifiers such as `content_id` for grouping/splitting rather than using them as predictive features. The decision is therefore made at the content-page level: each row is one page/item that could enter the review queue.


In [ ]:
from pathlib import Path
import pandas as pd

candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('../data/raw/content_refresh_anonymized.csv')]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(DATA_PATH)
lane_columns = ['content_id', 'client_id', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'trend_direction', 'trend_pct']
lane_df = df[lane_columns].copy()
print(f'Rows: {len(lane_df):,} | Columns shown: {lane_df.shape[1]}')
print('One row = one pseudonymized content page/item.')
display(lane_df.head(10))


## 5. Why ML beats a fixed rule here

A fixed rule is still an important baseline, and it may be good enough if the decision can be explained by one or two stable conditions. The reason to test ML is that page-priority decisions can combine several observed signals at once: visibility, CTR, average position, content age, update recency, and content length. Their useful combinations may not be captured well by a single if-statement.

ML earns its place only if it produces a better **validated ranking** than a simple baseline and the improvement is useful at the review capacity we care about. If a transparent rule performs as well or better, the rule may be the better operational choice.

This is therefore an ML/analysis problem because the goal is to test whether multiple observable signals provide enough additional information to improve a real prioritization decision — not because using a model is automatically better.


In [ ]:
# Show the candidate feature set while keeping the proxy-defining fields out of the model.
candidate_features = [
    'content_age_days', 'days_since_last_update', 'impressions_90d',
    'avg_position', 'ctr', 'word_count'
]
forbidden_as_features = ['trend_direction', 'trend_pct', 'content_id', 'client_id']
print('Candidate observable features:', candidate_features)
print('Excluded from features:', forbidden_as_features)


## Self-check

- [x] Task type is named: ranking / scoring
- [x] Target/proxy is named and its limitation is explained
- [x] Success metric is named: Precision@K
- [x] Unit of analysis is defined as one content page per row
- [x] Code loads the starter dataset and displays a real page-level dataframe
- [x] The reason to test ML versus a fixed rule is tied to the decision
- [x] No client names, URLs, or private queries are used
- [ ] **Run Runtime → Run all in Colab and save the executed notebook back to GitHub before submitting.**
